In [1]:
import os
import json
import pickle
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence

# File paths
DATA_DIR = "../data"
FEATURES_PATH = os.path.join(DATA_DIR, "features.pkl")
CAPTIONS_PATH = os.path.join(DATA_DIR, "cleaned_captions.json")
WORD_TO_IDX_PATH = os.path.join(DATA_DIR, "word_to_idx.json")
METADATA_PATH = os.path.join(DATA_DIR, "dataset_metadata.json")
TRAIN_SPLIT_PATH = os.path.join(DATA_DIR, "Flickr_8k.trainImages.txt")
VAL_SPLIT_PATH = os.path.join(DATA_DIR, "Flickr_8k.devImages.txt")

# 1. Load precomputed features
print("Loading features...")
with open(FEATURES_PATH, "rb") as f:
    features = pickle.load(f)

# 2. Load cleaned captions
print("Loading captions...")
with open(CAPTIONS_PATH, "r", encoding="utf-8") as f:
    captions = json.load(f)

# 3. Load word-to-index mapping and metadata
with open(WORD_TO_IDX_PATH, "r", encoding="utf-8") as f:
    word_to_idx = json.load(f)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

vocab_size = metadata["vocab_size"]
max_seq_len = metadata["max_seq_len"]

# 4. Load split image lists
def load_split_names(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

train_image_names = load_split_names(TRAIN_SPLIT_PATH)
val_image_names = load_split_names(VAL_SPLIT_PATH)

print(f"Vocab size: {vocab_size}")
print(f"Max sequence length: {max_seq_len}")
print(f"Training images: {len(train_image_names)}")
print(f"Validation images: {len(val_image_names)}")

I0000 00:00:1793361134.226578    2917 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading features...
Loading captions...
Vocab size: 2984
Max sequence length: 37
Training images: 6000
Validation images: 1000


In [2]:
class CaptionDataGenerator(Sequence):
    """
    High-Performance Data Generator for image-caption sequence models.
    - Pre-tokenizes captions once during __init__
    - Vectorized zero-padded array allocation (zero calls to pad_sequences)
    - Fast one-hot conversion via numpy vector indexing
    """
    def __init__(self, image_names, captions_dict, features_dict, word_to_idx, max_seq_len, vocab_size, batch_size=32, shuffle=True, to_sparse=False):
        # Keep only images with both features and captions available
        self.image_names = [name for name in image_names if name in features_dict and name in captions_dict]
        self.features_dict = features_dict
        self.max_seq_len = max_seq_len
        self.vocab_size = vocab_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.to_sparse = to_sparse

        # Infer feature dimension from first available sample
        first_feat = next(iter(features_dict.values()))
        self.feature_dim = first_feat.shape[0]

        # Pre-tokenize all captions once into integer IDs to eliminate runtime split() & dict lookups
        unk_id = word_to_idx.get("<unk>", 1)
        self.tokenized_captions = {}
        for name in self.image_names:
            self.tokenized_captions[name] = [
                [word_to_idx.get(w, unk_id) for w in cap.split()]
                for cap in captions_dict[name]
            ]

        self.indexes = np.arange(len(self.image_names))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.image_names) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        batch_names = [self.image_names[k] for k in batch_indexes]

        # 1. Compute exact sample count for this batch
        total_samples = sum(
            len(seq) - 1
            for name in batch_names
            for seq in self.tokenized_captions[name]
        )

        # 2. Pre-allocate NumPy buffers in one contiguous block
        X_img = np.empty((total_samples, self.feature_dim), dtype=np.float32)
        X_seq = np.zeros((total_samples, self.max_seq_len), dtype=np.int32)  # Zero-padded by default
        y = np.empty(total_samples, dtype=np.int32)

        # 3. Fast C-slice population (replaces ~1,700 calls to pad_sequences)
        pos = 0
        for name in batch_names:
            feat = self.features_dict[name]
            for seq in self.tokenized_captions[name]:
                n = len(seq)
                for i in range(1, n):
                    X_img[pos] = feat
                    X_seq[pos, :i] = seq[:i]   # Fast slice assignment; rest remains 0 (padded)
                    y[pos] = seq[i]
                    pos += 1

        # 4. Output targets
        if self.to_sparse:
            return [X_img, X_seq], y
        else:
            y_cat = np.eye(self.vocab_size, dtype=np.float32)[y]
            return [X_img, X_seq], y_cat

In [3]:
# Instantiate training and validation generators
train_generator = CaptionDataGenerator(
    image_names=train_image_names,
    captions_dict=captions,
    features_dict=features,
    word_to_idx=word_to_idx,
    max_seq_len=max_seq_len,
    vocab_size=vocab_size,
    batch_size=32,
    shuffle=True
)

val_generator = CaptionDataGenerator(
    image_names=val_image_names,
    captions_dict=captions,
    features_dict=features,
    word_to_idx=word_to_idx,
    max_seq_len=max_seq_len,
    vocab_size=vocab_size,
    batch_size=32,
    shuffle=False
)

# Benchmark fetching single batch
start = time.time()
(batch_x_img, batch_x_seq), batch_y = train_generator[0]
fetch_time_ms = (time.time() - start) * 1000

print("Batch shapes verification:")
print(f"  Image inputs shape:     {batch_x_img.shape}   (Batch Samples, Feature Dim)")
print(f"  Sequence inputs shape:  {batch_x_seq.shape}   (Batch Samples, Max Seq Len)")
print(f"  Target output shape:    {batch_y.shape}   (Batch Samples, Vocab Size)")
print(f"  Batch generation time:  {fetch_time_ms:.2f} ms")
print(f"\nTotal training batches per epoch: {len(train_generator)}")
print(f"Total validation batches per epoch: {len(val_generator)}")

Batch shapes verification:
  Image inputs shape:     (1859, 1920)   (Batch Samples, Feature Dim)
  Sequence inputs shape:  (1859, 37)   (Batch Samples, Max Seq Len)
  Target output shape:    (1859, 2984)   (Batch Samples, Vocab Size)
  Batch generation time:  744.88 ms

Total training batches per epoch: 187
Total validation batches per epoch: 31


In [4]:
# [Optional / Recommended for Training] Wrap generator into tf.data.Dataset with Prefetching
def create_tf_dataset(generator):
    """
    Wraps a CaptionDataGenerator into a tf.data.Dataset with prefetching.
    Overlaps CPU batch generation with GPU training steps, eliminating GPU idle time.
    """
    def gen_func():
        for i in range(len(generator)):
            yield generator[i]

    output_signature = (
        (
            tf.TensorSpec(shape=(None, generator.feature_dim), dtype=tf.float32),
            tf.TensorSpec(shape=(None, generator.max_seq_len), dtype=tf.int32)
        ),
        tf.TensorSpec(shape=(None, generator.vocab_size), dtype=tf.float32)
    )

    return tf.data.Dataset.from_generator(gen_func, output_signature=output_signature).prefetch(tf.data.AUTOTUNE)

# Create prefetched tf.data pipelines
train_ds = create_tf_dataset(train_generator)
val_ds = create_tf_dataset(val_generator)

print("Prefetched tf.data datasets ready for training!")

W0000 00:00:1793361330.778454    2917 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Prefetched tf.data datasets ready for training!
